# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

In [1]:
# ============================================================
# COLAB PROJECT SETUP
# ============================================================

from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount("/content/drive")

# Permanent project folder in Google Drive
DRIVE_PROJECT_DIR = Path(
    "/content/drive/MyDrive/Hybrid_RAG_Customer_Support"
)

# Temporary workspace for the current Colab runtime
LOCAL_PROJECT_DIR = Path(
    "/content/Hybrid_RAG_Customer_Support"
)

LOCAL_PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Work from the temporary Colab directory
os.chdir(LOCAL_PROJECT_DIR)

print("Google Drive project:", DRIVE_PROJECT_DIR)
print("Local Colab workspace:", LOCAL_PROJECT_DIR)
print("Current working directory:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive project: /content/drive/MyDrive/Hybrid_RAG_Customer_Support
Local Colab workspace: /content/Hybrid_RAG_Customer_Support
Current working directory: /content/Hybrid_RAG_Customer_Support


In [2]:
# ============================================================
# VERIFY GPU RUNTIME
# ============================================================

import torch

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU runtime is not enabled. In Colab, go to "
        "Runtime → Change runtime type → T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory_gb = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("GPU detected    :", gpu_name)
print(f"GPU memory      : {gpu_memory_gb:.2f} GB")
print("CUDA version    :", torch.version.cuda)
print("\nGPU runtime is enabled successfully.")

PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU detected    : Tesla T4
GPU memory      : 14.56 GB
CUDA version    : 12.8

GPU runtime is enabled successfully.


### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [3]:
# YOUR CODE HERE
!pip install -q -U \
    langchain-huggingface \
    langchain-community \
    langchain-text-splitters \
    sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [4]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

# Fixed embedding configuration
EMBEDDING_MODEL_ID = (
    "sentence-transformers/all-MiniLM-L6-v2"
)

EXPECTED_EMBEDDING_DIMENSION = 384
CHUNK_SIZE = 500
CHUNK_OVERLAP = 75

POLICY_DIR = (
    DRIVE_PROJECT_DIR
    / "data"
    / "corporate_policies"
)

if not POLICY_DIR.exists():
    raise FileNotFoundError(
        f"Corporate policy directory not found:\n{POLICY_DIR}"
    )

policy_files = sorted(POLICY_DIR.glob("*.md"))

if not policy_files:
    raise FileNotFoundError(
        f"No Markdown SOP files found in:\n{POLICY_DIR}"
    )

# Load each SOP as a LangChain Document
sop_documents = []

for file_path in policy_files:

    loader = TextLoader(
        str(file_path),
        encoding="utf-8"
    )

    loaded_documents = loader.load()

    for document in loaded_documents:

        if not document.page_content.strip():
            raise ValueError(
                f"Empty SOP document: {file_path.name}"
            )

        document.metadata.update({
            "source_file": file_path.name,
            "document_name": file_path.stem
        })

        sop_documents.append(document)

assert len(sop_documents) == len(policy_files)

print(f"SOP files loaded: {len(sop_documents)}")

/tmp/ipykernel_1550/746732061.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


SOP files loaded: 13


In [5]:
# Split SOPs into retrieval-ready chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n## ",
        "\n### ",
        "\n\n",
        "\n",
        " ",
        ""
    ],
    length_function=len
)

retrieval_chunks = text_splitter.split_documents(
    sop_documents
)

if not retrieval_chunks:
    raise ValueError(
        "No retrieval chunks were created."
    )


# Add stable chunk identifiers
chunk_counts = {}

for chunk in retrieval_chunks:

    source_file = chunk.metadata["source_file"]

    chunk_number = chunk_counts.get(
        source_file,
        0
    )

    chunk.metadata["chunk_id"] = (
        f"{Path(source_file).stem}_chunk_{chunk_number:03d}"
    )

    chunk_counts[source_file] = chunk_number + 1

# Initialise the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_ID,

    # CPU is sufficient for this small embedding model.
    # This leaves T4 memory available for the Qwen model later.
    model_kwargs={
        "device": "cpu"
    },

    # Normalised vectors support cosine-similarity retrieval.
    encode_kwargs={
        "normalize_embeddings": True
    }
)

print("Embedding model initialised successfully")
print(f"Embedding model: {EMBEDDING_MODEL_ID}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model initialised successfully
Embedding model: sentence-transformers/all-MiniLM-L6-v2


In [6]:
# Generate embeddings for every retrieval chunk
chunk_texts = [
    chunk.page_content
    for chunk in retrieval_chunks
]

document_embeddings = embedding_model.embed_documents(
    chunk_texts
)

embedding_array = np.asarray(
    document_embeddings,
    dtype=np.float32
)

# Validate a sample query embedding
sample_query = (
    "My order has not arrived within the expected delivery time."
)

sample_query_embedding = embedding_model.embed_query(
    sample_query
)

sample_query_embedding = np.asarray(
    sample_query_embedding,
    dtype=np.float32
)

# Validate embedding quality and dimensions
assert embedding_array.ndim == 2

assert embedding_array.shape[0] == len(
    retrieval_chunks
)

assert embedding_array.shape[1] == (
    EXPECTED_EMBEDDING_DIMENSION
)

assert sample_query_embedding.shape == (
    EXPECTED_EMBEDDING_DIMENSION,
)

assert np.isfinite(embedding_array).all()

assert np.isfinite(sample_query_embedding).all()

embedding_norms = np.linalg.norm(
    embedding_array,
    axis=1
)

assert np.all(embedding_norms > 0)

assert not np.allclose(
    embedding_array,
    0
)

In [7]:
# Create an embedding-validation summary
embedding_summary = pd.DataFrame({
    "chunk_id": [
        chunk.metadata["chunk_id"]
        for chunk in retrieval_chunks
    ],
    "source_file": [
        chunk.metadata["source_file"]
        for chunk in retrieval_chunks
    ],
    "character_count": [
        len(chunk.page_content)
        for chunk in retrieval_chunks
    ],
    "embedding_dimension": [
        len(vector)
        for vector in document_embeddings
    ],
    "embedding_norm": embedding_norms
})


print("\nEmbedding validation summary")
print("-" * 55)
print(f"Original SOP documents : {len(sop_documents)}")
print(f"Retrieval chunks       : {len(retrieval_chunks)}")
print(f"Embedding rows         : {embedding_array.shape[0]}")
print(f"Embedding dimensions   : {embedding_array.shape[1]}")
print(
    f"Sample query dimension : "
    f"{len(sample_query_embedding)}"
)
print(
    f"Minimum vector norm    : "
    f"{embedding_norms.min():.4f}"
)
print(
    f"Maximum vector norm    : "
    f"{embedding_norms.max():.4f}"
)
print(
    "Contains NaN/Inf       : "
    f"{not np.isfinite(embedding_array).all()}"
)

print("\nSample embedded chunks:")
display(
    embedding_summary.head()
)

print("\nSample query:")
print(sample_query)

print("\nFirst 10 values of its embedding:")
print(sample_query_embedding[:10])


Embedding validation summary
-------------------------------------------------------
Original SOP documents : 13
Retrieval chunks       : 65
Embedding rows         : 65
Embedding dimensions   : 384
Sample query dimension : 384
Minimum vector norm    : 1.0000
Maximum vector norm    : 1.0000
Contains NaN/Inf       : False

Sample embedded chunks:


,chunk_id,source_file,character_count,embedding_dimension,embedding_norm
0,account_recovery_chunk_000,account_recovery.md,286,384,1.0
1,account_recovery_chunk_001,account_recovery.md,291,384,1.0
2,account_recovery_chunk_002,account_recovery.md,292,384,1.0
3,account_recovery_chunk_003,account_recovery.md,239,384,1.0
4,account_recovery_chunk_004,account_recovery.md,436,384,1.0



Sample query:
My order has not arrived within the expected delivery time.

First 10 values of its embedding:
[-0.00752728 -0.00168537  0.03396824  0.02766392 -0.00038173 -0.08274966
 -0.04060418 -0.0347854   0.04040639  0.04041477]


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [8]:
# YOUR CODE HERE
constraints = """
requests==2.32.4
fsspec==2025.3.0
numpy>=1.26,<2.1
websockets>=15.0.1,<16
protobuf>=5.26.1,<6
rich>=10.11,<14
tokenizers>=0.22.0,<=0.23.0
opentelemetry-api==1.42.1
opentelemetry-sdk==1.42.1
opentelemetry-semantic-conventions==0.63b1
opentelemetry-exporter-otlp-proto-grpc==1.42.1
opentelemetry-exporter-otlp-proto-common==1.42.1
opentelemetry-proto==1.42.1
"""

with open("/tmp/colab_constraints.txt", "w") as file:
    file.write(constraints)

%pip install -q \
    --constraint /tmp/colab_constraints.txt \
    "chromadb==1.0.20" \
    "langchain-chroma==0.2.6" \
    "jedi>=0.16"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [9]:
from importlib.metadata import version
from langchain_chroma import Chroma
import chromadb

packages = [
    "chromadb",
    "langchain-chroma",
    "opentelemetry-api",
    "opentelemetry-sdk",
    "opentelemetry-semantic-conventions",
]

for package in packages:
    print(f"{package:<38}: {version(package)}")

print("\nChroma imported successfully.")

chromadb                              : 1.0.20
langchain-chroma                      : 0.2.6
opentelemetry-api                     : 1.42.1
opentelemetry-sdk                     : 1.42.1
opentelemetry-semantic-conventions    : 0.63b1

Chroma imported successfully.


In [10]:
import shutil
from langchain_chroma import Chroma

# Define the persistent vector-database location
NOTEBOOK_4_ARTIFACT_DIR = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_4"
)

CHROMA_DB_PATH = (
    NOTEBOOK_4_ARTIFACT_DIR
    / "chroma_db"
)

COLLECTION_NAME = "corporate_policy_chunks"

NOTEBOOK_4_ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Remove an older index so the notebook can be rerun cleanly
if CHROMA_DB_PATH.exists():
    shutil.rmtree(CHROMA_DB_PATH)

# Prepare stable IDs for every retrieval chunk
chunk_ids = [
    chunk.metadata["chunk_id"]
    for chunk in retrieval_chunks
]

assert len(chunk_ids) == len(retrieval_chunks)
assert len(chunk_ids) == len(set(chunk_ids))

In [12]:
# Create and persist the Chroma vector index
vector_db = Chroma.from_documents(
    documents=retrieval_chunks,
    embedding=embedding_model,
    ids=chunk_ids,
    collection_name=COLLECTION_NAME,
    persist_directory=str(CHROMA_DB_PATH),

    # Configure cosine distance for semantic retrieval
    collection_metadata={
        "hnsw:space": "cosine"
    }
)

# Validate the collection
indexed_record_count = vector_db._collection.count()
expected_record_count = len(retrieval_chunks)

assert indexed_record_count == expected_record_count

print("Chroma vector index created successfully")
print("-" * 55)
print(f"Collection name       : {COLLECTION_NAME}")
print(f"Persistent directory  : {CHROMA_DB_PATH}")
print(f"Expected chunk count  : {expected_record_count}")
print(f"Indexed chunk count   : {indexed_record_count}")
print(f"Embedding model       : {EMBEDDING_MODEL_ID}")
print("Distance configuration: cosine")

Chroma vector index created successfully
-------------------------------------------------------
Collection name       : corporate_policy_chunks
Persistent directory  : /content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_4/chroma_db
Expected chunk count  : 65
Indexed chunk count   : 65
Embedding model       : sentence-transformers/all-MiniLM-L6-v2
Distance configuration: cosine


In [13]:
# Run a test similarity search
TEST_SEARCH_QUERY = "refund"

test_results = vector_db.similarity_search_with_score(
    query=TEST_SEARCH_QUERY,
    k=3
)

assert len(test_results) == 3

test_result_summary = pd.DataFrame([
    {
        "rank": rank,
        "source_file": document.metadata.get("source_file"),
        "chunk_id": document.metadata.get("chunk_id"),
        "distance": float(distance),
        "content_preview": (
            document.page_content
            .replace("\n", " ")[:180]
        )
    }
    for rank, (document, distance) in enumerate(
        test_results,
        start=1
    )
])

print(f"\nTest query: {TEST_SEARCH_QUERY}")
print(
    "Lower cosine-distance values indicate "
    "closer semantic matches."
)

display(test_result_summary)


Test query: refund
Lower cosine-distance values indicate closer semantic matches.


,rank,source_file,chunk_id,distance,content_preview
0,1,refund_policy.md,refund_policy_chunk_000,0.414943,# Refund Policy ## Eligibility Customers may ...
1,2,refund_policy.md,refund_policy_chunk_001,0.428345,## Refund Methods Approved refunds are issued ...
2,3,refund_policy.md,refund_policy_chunk_004,0.444905,## Agent Steps 1. Verify the order identifier ...


In [14]:
# Reload the persisted index and validate persistence
reloaded_vector_db = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DB_PATH)
)

reloaded_count = reloaded_vector_db._collection.count()

assert reloaded_count == expected_record_count

reload_results = reloaded_vector_db.similarity_search(
    query=TEST_SEARCH_QUERY,
    k=1
)

assert len(reload_results) == 1

print("\nPersistence validation")
print("-" * 55)
print(f"Reloaded collection count: {reloaded_count}")
print(
    "Top source after reload  : "
    f"{reload_results[0].metadata.get('source_file')}"
)
print("Persistent Chroma index validated successfully.")


Persistence validation
-------------------------------------------------------
Reloaded collection count: 65
Top source after reload  : refund_policy.md
Persistent Chroma index validated successfully.


#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [15]:
# YOUR CODE HERE
%pip install -q bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 21.4 MB/s eta 0:00:00


In [16]:
import json

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)

# Load the baseline artefact created by Notebook 3
NOTEBOOK_3_OUTPUT_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_3"
    / "outputs.json"
)

if not NOTEBOOK_3_OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 3 outputs.json was not found at:\n"
        f"{NOTEBOOK_3_OUTPUT_PATH}"
    )

with open(
    NOTEBOOK_3_OUTPUT_PATH,
    "r",
    encoding="utf-8"
) as file:
    baseline_artifact = json.load(file)

required_fields = {
    "test_query",
    "ground_truth",
    "baseline_output"
}

missing_fields = (
    required_fields
    - set(baseline_artifact.keys())
)

if missing_fields:
    raise ValueError(
        f"outputs.json is missing required fields: "
        f"{sorted(missing_fields)}"
    )

test_query = baseline_artifact["test_query"]

print("Notebook 3 artefact loaded successfully")
print(f"\nRaw customer query:\n{test_query}")

Notebook 3 artefact loaded successfully

Raw customer query:
My order was supposed to arrive already, but it is still not here. What is happening and when will I receive it?


In [17]:
# Retrieve the top-1 SOP chunk using the raw query
TOP_K = 1

# Use the persisted and reloaded Chroma index from Task 3.2.2
search_db = (
    reloaded_vector_db
    if "reloaded_vector_db" in globals()
    else vector_db
)

retrieved_documents = search_db.similarity_search(
    query=test_query,
    k=TOP_K
)

if not retrieved_documents:
    raise RuntimeError(
        "The vector database returned no documents."
    )

retrieved_document = retrieved_documents[0]

retrieved_context = (
    retrieved_document.page_content.strip()
)

retrieved_source = (
    retrieved_document.metadata.get(
        "source_file",
        "unknown"
    )
)

retrieved_chunk_id = (
    retrieved_document.metadata.get(
        "chunk_id",
        "unknown"
    )
)

if not retrieved_context:
    raise ValueError(
        "The retrieved SOP chunk is empty."
    )

print("\nTop-1 retrieval result")
print("-" * 60)
print(f"Retrieved source   : {retrieved_source}")
print(f"Retrieved chunk ID : {retrieved_chunk_id}")
print("\nRetrieved context:")
print(retrieved_context)


Top-1 retrieval result
------------------------------------------------------------
Retrieved source   : order_tracking.md
Retrieved chunk ID : order_tracking_chunk_003

Retrieved context:
## "Delivered But Not Received"
1. Confirm the shipping address on the order.
2. Ask the customer to check with neighbors, household members, and safe
   drop-off spots, and to allow 24 hours, as carriers sometimes scan early.
3. If still missing, open a carrier investigation and offer a replacement or
   refund per the shipping delays and refund procedures.


In [18]:
# Load the same baseline model used in Notebook 3
MODEL_ID = baseline_artifact.get(
    "model_id",
    "Qwen/Qwen2.5-1.5B-Instruct"
)

MAX_NEW_TOKENS = 120

if not torch.cuda.is_available():
    raise RuntimeError(
        "T4 GPU runtime is required for model inference."
    )

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    use_fast=True
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

model.eval()

print("\nGeneration model loaded successfully")
print(f"Model ID : {MODEL_ID}")
print(f"GPU      : {torch.cuda.get_device_name(0)}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Generation model loaded successfully
Model ID : Qwen/Qwen2.5-1.5B-Instruct
GPU      : Tesla T4


In [19]:
# Augment the prompt with the retrieved SOP context
system_prompt = (
    "You are a customer-support assistant. "
    "Answer strictly using only the supplied SOP context. "
    "Do not invent timelines, procedures, departments, contact "
    "details, refunds, or guarantees. If the context does not "
    "contain enough information, state that the available policy "
    "does not provide enough information."
)

user_prompt = f"""
Customer query:
{test_query}

Retrieved SOP source:
{retrieved_source}

Retrieved SOP context:
{retrieved_context}

Provide a concise policy-grounded answer to the customer.
""".strip()

messages = [
    {
        "role": "system",
        "content": system_prompt
    },
    {
        "role": "user",
        "content": user_prompt
    }
]

prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer(
    prompt_text,
    return_tensors="pt",
    add_special_tokens=False
).to(model.device)

# 5. Generate the deterministic naive-RAG response
with torch.inference_mode():
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

new_token_ids = generated_ids[
    0,
    model_inputs["input_ids"].shape[1]:
]

naive_rag_output = tokenizer.decode(
    new_token_ids,
    skip_special_tokens=True
).strip()

if not naive_rag_output:
    raise RuntimeError(
        "The model generated an empty response."
    )

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [20]:
# Display and retain the workflow result
naive_rag_result = {
    "test_query": test_query,
    "retrieved_source": retrieved_source,
    "retrieved_chunk_id": retrieved_chunk_id,
    "retrieved_context": retrieved_context,
    "naive_rag_output": naive_rag_output
}

print("\nNaive RAG workflow result")
print("=" * 70)
print(f"Customer query:\n{test_query}")

print("\nRetrieved policy:")
print(retrieved_source)

print("\nNaive RAG output:")
print(naive_rag_output)


Naive RAG workflow result
Customer query:
My order was supposed to arrive already, but it is still not here. What is happening and when will I receive it?

Retrieved policy:
order_tracking.md

Naive RAG output:
Based on our tracking system, your order has been scanned by the carrier at an earlier time than expected. We recommend checking with your neighbors, household members, or local pickup locations within the next 24 hours for any possible delivery. If you have not received your order after this period, please follow up through our support channels. We apologize for any inconvenience caused and assure you that we are actively investigating the situation. Please expect a replacement or refund if applicable according to our shipping delays and refund policies. Thank you for understanding.


### Retrieval Workflow Observation

The Naive RAG pipeline successfully retrieved the top-ranked SOP chunk using the raw customer query and generated a context-augmented response. The retrieved source and generated output have been saved for formal evaluation in the Solution V1 RAG Evaluation notebook.

---
## Save Artifacts for Downstream Notebooks

In [21]:
# YOUR CODE HERE
import os

# Validate the persisted Chroma index
if not CHROMA_DB_PATH.exists():
    raise FileNotFoundError(
        f"Persisted Chroma index was not found at:\n{CHROMA_DB_PATH}"
    )

chroma_files = [
    path
    for path in CHROMA_DB_PATH.rglob("*")
    if path.is_file()
]

if not chroma_files:
    raise RuntimeError(
        f"The Chroma directory exists but contains no files:\n"
        f"{CHROMA_DB_PATH}"
    )

# Load Notebook 3's existing outputs.json
OUTPUT_PATH = (
    DRIVE_PROJECT_DIR
    / "artifacts"
    / "notebook_3"
    / "outputs.json"
)

if not OUTPUT_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 3 outputs.json was not found at:\n{OUTPUT_PATH}"
    )

with open(OUTPUT_PATH, "r", encoding="utf-8") as file:
    downstream_outputs = json.load(file)

In [22]:
# Add the naive-RAG result and retrieval metadata
downstream_outputs.update({
    "naive_rag_output": naive_rag_output,
    "naive_rag_retrieval": {
        "retrieved_source": retrieved_source,
        "retrieved_chunk_id": retrieved_chunk_id,
        "retrieved_context": retrieved_context,
        "top_k": TOP_K,
        "embedding_model_id": EMBEDDING_MODEL_ID,
        "collection_name": COLLECTION_NAME,
        "chroma_db_path": str(CHROMA_DB_PATH)
    }
})

# Save atomically to avoid a partially written JSON file
TEMP_OUTPUT_PATH = OUTPUT_PATH.with_suffix(".tmp")

with open(TEMP_OUTPUT_PATH, "w", encoding="utf-8") as file:
    json.dump(
        downstream_outputs,
        file,
        indent=2,
        ensure_ascii=False
    )

os.replace(TEMP_OUTPUT_PATH, OUTPUT_PATH)

In [23]:
# Reload and validate the saved artefact
with open(OUTPUT_PATH, "r", encoding="utf-8") as file:
    saved_outputs = json.load(file)

required_fields = {
    "test_query",
    "ground_truth",
    "baseline_output",
    "naive_rag_output",
    "naive_rag_retrieval"
}

missing_fields = required_fields - set(saved_outputs)

assert not missing_fields, (
    f"Missing required saved fields: {sorted(missing_fields)}"
)

assert saved_outputs["naive_rag_output"] == naive_rag_output
assert (
    saved_outputs["naive_rag_retrieval"]["retrieved_source"]
    == retrieved_source
)

# Confirmation
print("Notebook 4 artefacts saved successfully")
print("-" * 55)
print(f"Persisted Chroma index : {CHROMA_DB_PATH}")
print(f"Chroma files found     : {len(chroma_files)}")
print(f"Updated outputs.json   : {OUTPUT_PATH}")
print(f"Retrieved source       : {retrieved_source}")
print(f"Retrieved chunk        : {retrieved_chunk_id}")
print(f"Saved fields           : {list(saved_outputs.keys())}")

Notebook 4 artefacts saved successfully
-------------------------------------------------------
Persisted Chroma index : /content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_4/chroma_db
Chroma files found     : 5
Updated outputs.json   : /content/drive/MyDrive/Hybrid_RAG_Customer_Support/artifacts/notebook_3/outputs.json
Retrieved source       : order_tracking.md
Retrieved chunk        : order_tracking_chunk_003
Saved fields           : ['model_id', 'test_query', 'ground_truth', 'baseline_output', 'evaluation', 'naive_rag_output', 'naive_rag_retrieval']


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [x] SOP documents loaded via `TextLoader`
- [x] **Embeddings generated and validated** ← _Task 3.2.1_
- [x] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [x] Naive similarity search executed on `test_query`
- [x] Naive RAG output generated with SOP context injected
- [x] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [x] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**